<a href="https://colab.research.google.com/github/Sayantankhan/colabNoteBook/blob/main/Tokenizer_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import re
import urllib.request

if not os.path.exists("the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    file_path = "the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)

with open("/content/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


- The goal is to tokenize and embed this text for an LLM
- develop a simple tokenizer based on some simple sample text

In [ ]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])
print(len(preprocessed))

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']
4690


### Converting tokens into token IDs
- From these tokens, we can now build a vocabulary that consists of all the unique tokens

In [ ]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
vocab = {token:integer for integer, token in enumerate(all_words)}
print(vocab_size)
print(vocab)

1130
{'!': 0, '"': 1, "'": 2, '(': 3, ')': 4, ',': 5, '--': 6, '.': 7, ':': 8, ';': 9, '?': 10, 'A': 11, 'Ah': 12, 'Among': 13, 'And': 14, 'Are': 15, 'Arrt': 16, 'As': 17, 'At': 18, 'Be': 19, 'Begin': 20, 'Burlington': 21, 'But': 22, 'By': 23, 'Carlo': 24, 'Chicago': 25, 'Claude': 26, 'Come': 27, 'Croft': 28, 'Destroyed': 29, 'Devonshire': 30, 'Don': 31, 'Dubarry': 32, 'Emperors': 33, 'Florence': 34, 'For': 35, 'Gallery': 36, 'Gideon': 37, 'Gisburn': 38, 'Gisburns': 39, 'Grafton': 40, 'Greek': 41, 'Grindle': 42, 'Grindles': 43, 'HAD': 44, 'Had': 45, 'Hang': 46, 'Has': 47, 'He': 48, 'Her': 49, 'Hermia': 50, 'His': 51, 'How': 52, 'I': 53, 'If': 54, 'In': 55, 'It': 56, 'Jack': 57, 'Jove': 58, 'Just': 59, 'Lord': 60, 'Made': 61, 'Miss': 62, 'Money': 63, 'Monte': 64, 'Moon-dancers': 65, 'Mr': 66, 'Mrs': 67, 'My': 68, 'Never': 69, 'No': 70, 'Now': 71, 'Nutley': 72, 'Of': 73, 'Oh': 74, 'On': 75, 'Once': 76, 'Only': 77, 'Or': 78, 'Perhaps': 79, 'Poor': 80, 'Professional': 81, 'Renaissance': 82

In [ ]:
# Building a Simple Tokenizer
class TokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {v:k for k, v in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [ ]:
tokenizer = TokenizerV1(vocab)
print(tokenizer.int_to_str)


text = """"It's the last he painted, you know,"
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

{0: '!', 1: '"', 2: "'", 3: '(', 4: ')', 5: ',', 6: '--', 7: '.', 8: ':', 9: ';', 10: '?', 11: 'A', 12: 'Ah', 13: 'Among', 14: 'And', 15: 'Are', 16: 'Arrt', 17: 'As', 18: 'At', 19: 'Be', 20: 'Begin', 21: 'Burlington', 22: 'But', 23: 'By', 24: 'Carlo', 25: 'Chicago', 26: 'Claude', 27: 'Come', 28: 'Croft', 29: 'Destroyed', 30: 'Devonshire', 31: 'Don', 32: 'Dubarry', 33: 'Emperors', 34: 'Florence', 35: 'For', 36: 'Gallery', 37: 'Gideon', 38: 'Gisburn', 39: 'Gisburns', 40: 'Grafton', 41: 'Greek', 42: 'Grindle', 43: 'Grindles', 44: 'HAD', 45: 'Had', 46: 'Hang', 47: 'Has', 48: 'He', 49: 'Her', 50: 'Hermia', 51: 'His', 52: 'How', 53: 'I', 54: 'If', 55: 'In', 56: 'It', 57: 'Jack', 58: 'Jove', 59: 'Just', 60: 'Lord', 61: 'Made', 62: 'Miss', 63: 'Money', 64: 'Monte', 65: 'Moon-dancers', 66: 'Mr', 67: 'Mrs', 68: 'My', 69: 'Never', 70: 'No', 71: 'Now', 72: 'Nutley', 73: 'Of', 74: 'Oh', 75: 'On', 76: 'Once', 77: 'Only', 78: 'Or', 79: 'Perhaps', 80: 'Poor', 81: 'Professional', 82: 'Renaissance', 83:

### Adding special context tokens

It's useful to add some "special" tokens for unknown words and to denote the end of a text

- [BOS] (beginning of sequence) marks the beginning of text
- [EOS] (end of sequence) marks where the text ends
- [UNK] to represent words that are not included in the vocabulary. GPT uses BPE (byte pair tokenizer)
-  <|endoftext|> GPT uses for padding, reduce complexity and so on

In [ ]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

print(len(vocab))
vocab = {token:integer for integer,token in enumerate(all_tokens)}
print(len(vocab))

# printing last 5
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

1130
1132
('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [ ]:
class TokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {v: k for k, v in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]

        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text) # drop the spaces before punctuation
        return text

In [ ]:
tokenizer2 = TokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))
ids = tokenizer2.encode(text)
# print(ids)
print(text)
print(tokenizer2.decode(tokenizer2.encode(text)))

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


### Byte Pair Encoding (BPE)
- It allows the model to break down words that aren't in its predefined vocabulary into smaller subword units or even individual characters, enabling it to handle out-of-vocabulary words

- For instance, if GPT-2's vocabulary doesn't have the word "unfamiliarword," it might tokenize it as ["unfam", "iliar", "word"] or some other subword breakdown, depending on its trained BPE merges

- https://github.com/openai/gpt-2/blob/master/src/encoder.py

-  we are using the BPE tokenizer from OpenAI's open-source tiktoken library, which implements its core algorithms in Rust to improve computational performance

In [ ]:
"""Byte pair encoding utilities - OPENAI BPE Tokenizer"""

import os
import json
import regex as re
from functools import lru_cache
import requests
from tqdm import tqdm

@lru_cache()
def bytes_to_unicode():
    """
    Returns list of utf-8 byte and a corresponding list of unicode strings.
    The reversible bpe codes work on unicode strings.
    This means you need a large # of unicode characters in your vocab if you want to avoid UNKs.
    When you're at something like a 10B token dataset you end up needing around 5K for decent coverage.
    This is a signficant percentage of your normal, say, 32K bpe vocab.
    To avoid that, we want lookup tables between utf-8 bytes and unicode strings.
    And avoids mapping to whitespace/control characters the bpe code barfs on.
    """
    bs = list(range(ord("!"), ord("~")+1))+list(range(ord("¡"), ord("¬")+1))+list(range(ord("®"), ord("ÿ")+1))
    cs = bs[:]
    n = 0
    for b in range(2**8):
        if b not in bs:
            bs.append(b)
            cs.append(2**8+n)
            n += 1
    cs = [chr(n) for n in cs]
    return dict(zip(bs, cs))

def get_pairs(word):
    """Return set of symbol pairs in a word.

    Word is represented as tuple of symbols (symbols being variable-length strings).
    """
    pairs = set()
    prev_char = word[0]
    for char in word[1:]:
        pairs.add((prev_char, char))
        prev_char = char
    return pairs

class Encoder:
    def __init__(self, encoder, bpe_merges, errors='replace'):
        self.encoder = encoder
        self.decoder = {v:k for k,v in self.encoder.items()}
        self.errors = errors # how to handle errors in decoding
        self.byte_encoder = bytes_to_unicode()
        self.byte_decoder = {v:k for k, v in self.byte_encoder.items()}
        self.bpe_ranks = dict(zip(bpe_merges, range(len(bpe_merges))))
        self.cache = {}

        # Should haved added re.IGNORECASE so BPE merges can happen for capitalized versions of contractions
        self.pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

    def bpe(self, token):
        if token in self.cache:
            return self.cache[token]
        word = tuple(token)
        pairs = get_pairs(word)

        if not pairs:
            return token

        while True:
            bigram = min(pairs, key = lambda pair: self.bpe_ranks.get(pair, float('inf')))
            if bigram not in self.bpe_ranks:
                break
            first, second = bigram
            new_word = []
            i = 0
            while i < len(word):
                try:
                    j = word.index(first, i)
                    new_word.extend(word[i:j])
                    i = j
                except:
                    new_word.extend(word[i:])
                    break

                if word[i] == first and i < len(word)-1 and word[i+1] == second:
                    new_word.append(first+second)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_word = tuple(new_word)
            word = new_word
            if len(word) == 1:
                break
            else:
                pairs = get_pairs(word)
        word = ' '.join(word)
        self.cache[token] = word
        return word

    def encode(self, text):
        bpe_tokens = []
        for token in re.findall(self.pat, text):
            token = ''.join(self.byte_encoder[b] for b in token.encode('utf-8'))
            bpe_tokens.extend(self.encoder[bpe_token] for bpe_token in self.bpe(token).split(' '))
        return bpe_tokens

    def decode(self, tokens):
        text = ''.join([self.decoder[token] for token in tokens])
        text = bytearray([self.byte_decoder[c] for c in text]).decode('utf-8', errors=self.errors)
        return text

def get_encoder(model_name, models_dir):
    with open(os.path.join(models_dir, model_name, 'encoder.json'), 'r') as f:
        encoder = json.load(f)
    with open(os.path.join(models_dir, model_name, 'vocab.bpe'), 'r', encoding="utf-8") as f:
        bpe_data = f.read()
    bpe_merges = [tuple(merge_str.split()) for merge_str in bpe_data.split('\n')[1:-1]]
    return Encoder(
        encoder=encoder,
        bpe_merges=bpe_merges,
    )

def download_vocab():
    # Modified code from
    subdir = 'gpt2_model'
    if not os.path.exists(subdir):
        os.makedirs(subdir)
    subdir = subdir.replace('\\', '/')  # needed for Windows

    for filename in ['encoder.json', 'vocab.bpe']:
        r = requests.get("https://openaipublic.blob.core.windows.net/gpt-2/models/117M/" + filename, stream=True)

        with open(os.path.join(subdir, filename), 'wb') as f:
            file_size = int(r.headers["content-length"])
            chunk_size = 1000
            with tqdm(ncols=100, desc="Fetching " + filename, total=file_size, unit_scale=True) as pbar:
                # 1k for chunk_size, since Ethernet packet size is around 1500 bytes
                for chunk in r.iter_content(chunk_size=chunk_size):
                    f.write(chunk)
                    pbar.update(chunk_size)

In [ ]:
download_vocab()
bpe_encoder = get_encoder("gpt2_model", "/content")

print(tokenizer2.decode(tokenizer2.encode(text)))
print(bpe_encoder.decode(bpe_encoder.encode(text)))

Fetching encoder.json: 1.04Mit [00:00, 2.25Mit/s]                                                   
Fetching vocab.bpe: 457kit [00:00, 1.56Mit/s]                                                       


<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.
Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [ ]:
### BPE with tiktoken
!pip install tiktoken

import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))
tiktoken_gpt_tokenizer = tiktoken.get_encoding("gpt2")

tiktoken version: 0.11.0


In [ ]:
# BPE via Hugging Face transformers
!pip install transformers>=4.33.2
from transformers import GPT2Tokenizer
hf_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [ ]:
with open("/content/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()


%timeit tokenizer2.decode(tokenizer2.encode(raw_text))
%timeit bpe_encoder.decode(bpe_encoder.encode(raw_text))
%timeit tiktoken_gpt_tokenizer.decode(tiktoken_gpt_tokenizer.encode(raw_text, allowed_special={"<|endoftext|>"}))
%timeit hf_tokenizer(raw_text)["input_ids"]

11 ms ± 1.07 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)
30.9 ms ± 10.6 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


Token indices sequence length is longer than the specified maximum sequence length for this model (5145 > 1024). Running this sequence through the model will result in indexing errors


7.04 ms ± 3.05 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)
50.2 ms ± 3.62 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


### Data sampling - Sliding window

- We train LLMs to generate one word at a time, so we want to prepare the training data accordingly where the next word in a sequence represents the target to predict


In [ ]:
enc_text = tiktoken_gpt_tokenizer.encode(raw_text)
print(len(enc_text))

5145


- For each text chunk, we want the inputs and targets

- Since we want the model to predict the next word, the targets are the inputs shifted by one position to the right

In [ ]:
enc_sample = enc_text[50:]

context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y: {y}")

for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

x: [290, 4920, 2241, 287]
y: [4920, 2241, 287, 257]
[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [ ]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tiktoken_gpt_tokenizer.decode(context), "---->", tiktoken_gpt_tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


we implement a simple data loader that iterates over the input dataset and returns the inputs and targets shifted by one

In [ ]:
import torch
print("PyTorch version:", torch.__version__)

# Create dataset and dataloader that extract chunks from the input text dataset
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length + 1, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def create_dataloader_v1(text, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    """
        Stride = the step size you move by when applying a sliding operation.
    """
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(text, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader


PyTorch version: 2.8.0+cu126


In [ ]:
# Test dataloader
with open("/content/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

second_batch = next(data_iter)
print(second_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [ ]:
# We can also create batched outputs
# Note that we increase the stride here so that we don't have overlaps between the batches, since more overlap could lead to increased overfitting

dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


### Creating token embeddings

- The data is already almost ready for an LLM

-  lastly let us embed the tokens in a continuous vector representation using an embedding layer

- Usually, these embedding layers are part of the LLM itself and are updated (trained) during model training

![Stride Example](https://camo.githubusercontent.com/cad0f97a119e344cc132e00a75f4b29baa23bae26eb9efa31fcb2ff44bc8b1cc/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f31352e77656270)


- For those who are familiar with one-hot encoding, the embedding layer approach above is essentially just a more efficient way of implementing one-hot encoding followed by matrix multiplication in a fully-connected laye

- Because the embedding layer is just a more efficient implementation that is equivalent to the one-hot encoding and matrix-multiplication approach it can be seen as a neural network layer that can be optimized via backpropagation

In [ ]:
# Suppose we have the following four input examples with input ids 2, 3, 5, and 1 (after tokenization):
input_ids = torch.tensor([2, 3, 5, 1])


# For the sake of simplicity, suppose we have a small vocabulary of only 6 words and we want to create embeddings of size 3:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight) # This would result in a 6x3 weight matrix:

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [ ]:
# To convert a token with id 3 into a 3-dimensional vector, we do the following:
print(embedding_layer(torch.tensor([3])))

# NOTE : the 4th row in the embedding_layer weight matrix
# To embed all four input_ids values above
print(embedding_layer(input_ids))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


![Embedding](https://camo.githubusercontent.com/30c75dce5178bdb6f53a37899c08b44c92eff2306c38beef19a831dc3770fc00/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f31362e776562703f313233)

Embedding layer convert IDs into identical vector representations regardless of where they are located in the input sequence

Positional embeddings are combined with the token embedding vector to form the input embeddings for a large language model:

The BytePair encoder has a vocabulary size of 50,257. Suppose we want to encode the input tokens into a 256-dimensional vector representation

In [ ]:
vocab_size = 50257
output_dim = 256


- If we sample data from the dataloader, we embed the tokens in each batch into a 256-dimensional vector
- If we have a batch size of 8 with 4 tokens each, this results in a 8 x 4 x 256 tensor:

In [ ]:

max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("==== Input ====")
print("Token IDs:\n", inputs)
print("==== Target ====")
print("Token IDs:\n", targets)
print("\nInputs shape:\n", inputs.shape)

==== Input ====
Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
==== Target ====
Token IDs:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])

Inputs shape:
 torch.Size([8, 4])


In [ ]:
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

# GPT-2 uses absolute position embeddings, so we just create another embedding layer:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)


# To create the input embeddings used in an LLM, we simply add the token and the positional embeddings:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)


torch.Size([8, 4, 256])
torch.Size([4, 256])
tensor([[[ 0.5762,  2.2692,  0.3580,  ..., -0.4424, -1.4077, -0.6379],
         [ 0.7154,  0.1471, -0.6837,  ...,  0.2181,  1.4199, -1.2985],
         [ 0.0531,  1.2028,  0.4994,  ..., -0.2618,  1.2670,  0.5114],
         [ 0.1393, -0.9740,  0.0821,  ...,  0.4408, -1.2866, -0.7939]],

        [[-0.6552, -0.6020,  2.1738,  ...,  0.8536, -0.2478,  0.2526],
         [ 0.1769,  1.6029, -0.4450,  ...,  1.3989,  0.0481,  0.0310],
         [-0.5414, -1.0289,  1.1124,  ..., -1.0645, -0.8940, -0.0789],
         [-0.8905,  0.3382,  0.5494,  ..., -0.0467,  0.7027, -2.1751]],

        [[ 0.7106, -1.4915,  1.5205,  ..., -0.6312,  0.4474,  0.1313],
         [ 2.3435,  2.2140, -0.6660,  ..., -1.2622, -1.1711,  0.3457],
         [-1.9431, -0.6429,  1.4388,  ...,  3.1715,  1.3620, -0.8376],
         [ 0.4197, -0.6103,  0.3624,  ...,  1.0081,  0.2992, -1.7300]],

        ...,

        [[-0.4520, -1.5876, -0.6390,  ..., -0.3110,  1.0419,  0.4355],
         [-1

![Image](https://camo.githubusercontent.com/730badacd85e476130cab5a98990d3c616b4333921096c576c31a50e7c0ca627/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f31392e77656270)

In [5]:
# Clean CURRENT Colab notebook metadata in-place (remove widgets block)
# Then use: File → Save a copy in GitHub

# Force-normalize metadata.widgets for GitHub rendering
# Then use: File → Save a copy in GitHub

import json
from google.colab import _message

MIME_KEY = "application/vnd.jupyter.widget-state+json"

def get_ipynb():
    resp = _message.blocking_request('get_ipynb', timeout_sec=10)
    nb = resp.get('ipynb', resp)  # handle both wrapped and direct forms
    if isinstance(nb, str):
        nb = json.loads(nb)
    return nb

def ensure_state(bundle):
    """Ensure a single widget-state bundle has 'state' and version fields."""
    changed = False
    if isinstance(bundle, str):
        try:
            b2 = json.loads(bundle); bundle = b2; changed = True
        except Exception:
            return bundle, changed
    if isinstance(bundle, dict):
        if "version_major" not in bundle: bundle["version_major"] = 2; changed = True
        if "version_minor" not in bundle: bundle["version_minor"] = 0; changed = True
        if "state" not in bundle:         bundle["state"] = {};          changed = True
    return bundle, changed

def normalize_widgets(widgets):
    """Handle multiple possible shapes Colab may produce."""
    changed = False
    if not isinstance(widgets, dict):
        # If widgets is junk, replace with a minimal valid bundle
        return {MIME_KEY: {"version_major": 2, "version_minor": 0, "state": {}}}, True

    # Case 1: Standard single-bundle under MIME_KEY
    if MIME_KEY in widgets:
        widgets[MIME_KEY], c = ensure_state(widgets[MIME_KEY])
        return widgets, changed or c

    # Case 2: Other keys present (older/odd Colab shapes) – fix each dict-like value
    any_fixed = False
    for k, v in list(widgets.items()):
        if isinstance(v, (dict, str)):
            fixed, c = ensure_state(v)
            widgets[k] = fixed
            any_fixed = any_fixed or c

    # If none had the MIME_KEY, add a minimal valid one so GitHub is happy
    if MIME_KEY not in widgets:
        widgets[MIME_KEY] = {"version_major": 2, "version_minor": 0, "state": {}}
        any_fixed = True

    return widgets, any_fixed

def set_ipynb(nb):
    _message.blocking_request('set_ipynb', {'ipynb': nb})

# --- Apply fix ---
nb = get_ipynb()
meta = nb.setdefault("metadata", {})
widgets = meta.get("widgets", {})

widgets_fixed, changed = normalize_widgets(widgets)
meta["widgets"] = widgets_fixed
nb["metadata"] = meta

set_ipynb(nb)

# --- Verify (hard fail if still invalid) ---
nb2 = get_ipynb()
w2 = nb2.get("metadata", {}).get("widgets", {})
ok = (isinstance(w2, dict) and MIME_KEY in w2
      and isinstance(w2[MIME_KEY], dict) and "state" in w2[MIME_KEY])

print("Verification:", "OK ✅" if ok else "STILL INVALID ❌")
if not ok:
    print("Debug dump (truncated):", json.dumps(w2)[:800])
else:
    st = w2[MIME_KEY]["state"]
    print(f"State keys: {len(st) if isinstance(st, dict) else 'n/a'} (should be dict)")
print("Now use: File → Save a copy in GitHub.")



Verification: STILL INVALID ❌
Debug dump (truncated): {"application/vnd.jupyter.widget-state+json": {"40be92f324ef44e5bcd502fc8e9010fe": {"model_module": "@jupyter-widgets/controls", "model_name": "HBoxModel", "model_module_version": "1.5.0", "state": {"_dom_classes": [], "_model_module": "@jupyter-widgets/controls", "_model_module_version": "1.5.0", "_model_name": "HBoxModel", "_view_count": null, "_view_module": "@jupyter-widgets/controls", "_view_module_version": "1.5.0", "_view_name": "HBoxView", "box_style": "", "children": ["IPY_MODEL_b466493d5a3446e3ae5fa82ca73837de", "IPY_MODEL_e77430220d3a487ca39f22ddaf1eafd1", "IPY_MODEL_6de42b519ed74fda89d16f0d3f9343e3"], "layout": "IPY_MODEL_9eac7ae241f34dc3b80ae15255eef316"}}, "b466493d5a3446e3ae5fa82ca73837de": {"model_module": "@jupyter-widgets/controls", "model_name": "HTMLModel", "model_modul
Now use: File → Save a copy in GitHub.
